# 03 — Tomography & Fidelity on Real Hardware

Counts hand you probabilities — one number per outcome. But the device prepared a *whole* state, with phases and coherences that a single histogram cannot see. To recover all of it you measure in three complementary bases and reconstruct the state, then ask the honest question: how far did it fall from the ideal you asked for?

**Prerequisites:** `01/14_fidelity_and_trace_distance`, `01/15_tomography_of_a_noisy_qubit`, `05/01_using_a_real_quantum_computer`.

**What you'll be able to do**
- Run single-qubit Pauli tomography through the runtime primitives (`SamplerV2`), one circuit per basis.
- Reconstruct the device's density matrix `rho_hat` from the counts with `density_from_counts`.
- Read a purity below 1 and a fidelity below 1, and name what each one measures.
- Add a new row to the classical-to-quantum dictionary: the *ideal* `rho` you asked for versus the *device* `rho_hat` you got back — pure giving way to mixed.

This is the primitive-path successor to `01/15`, where you ran tomography with a local `backend.run`. The mathematics is identical; what changes is that the same code now reaches a real QPU through `SamplerV2`. You already know how to turn counts into a state — here you turn a *device's* counts into the state it actually prepared, and then you measure the gap. Reconstructing a real quantum state from nothing but click statistics is a genuine milestone, so let us go and do it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2
from qot_course.hardware.runtime import select_backend
from qot_course.hardware.tomography import single_qubit_tomography_circuits, density_from_counts
from qot_course.quantum.density import purity, fidelity, density_matrix
from qot_course.quantum.states import KET_PLUS

np.random.seed(42)
viz.use_course_style()

In [ ]:
USE_HARDWARE = False  # flip to True after you've set up credentials (see 05/01)
backend, label, is_real = select_backend(use_hardware=USE_HARDWARE)
print(f"Running on: {label}  (real hardware = {is_real})")

## 1 · Measuring in three bases

A single qubit's state is fixed completely by three numbers: the expectation values $\langle X \rangle$, $\langle Y \rangle$, and $\langle Z \rangle$, which together are its Bloch vector. A measurement only ever reads the $Z$ axis, so to learn the other two we rotate them onto $Z$ first: one circuit measures $\langle Z \rangle$ directly, another rotates $X$ onto $Z$ before measuring, the third rotates $Y$. Three circuits, three Bloch components, one reconstructed state. The helper `single_qubit_tomography_circuits` builds all three for us — it takes the state-preparation circuit and returns a dictionary keyed `"X"`, `"Y"`, `"Z"`.

In [ ]:
prep = QuantumCircuit(1)
prep.h(0)                                   # the |+> we try to prepare
circuits = single_qubit_tomography_circuits(prep)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
settings = list(circuits)                   # ["X", "Y", "Z"]
isa_list = [pm.run(circuits[s]) for s in settings]
res = SamplerV2(mode=backend).run(isa_list, shots=4096).result()  # sampling is stochastic -- exact purity/fidelity shift run to run
counts_by_setting = {s: res[i].data.c.get_counts() for i, s in enumerate(settings)}

rho_hat = density_from_counts(counts_by_setting, n_qubits=1)
ideal = density_matrix(KET_PLUS)
print("purity(rho_hat)       :", round(purity(rho_hat), 3), " (< 1  =>  mixed: decoherence + finite-shot bias)")
print("fidelity(rho_hat, |+>):", round(fidelity(rho_hat, ideal), 3), " (< 1  =>  imperfect prep)")

In [ ]:
viz.plot_density_matrix(rho_hat, title="Tomography of |+> on the device", basis_labels=["|0>", "|1>"])
plt.show()

**Read the figure.** The reconstructed `rho_hat` has damped off-diagonal coherences compared with the ideal $|+\rangle\langle+|$, where every entry would sit at exactly $0.5$. Those shrunken corners, together with a purity below 1, are *imperfection* captured in a single picture — decoherence together with finite-shot and projection bias. Both effects pull reconstructed purity below 1: even a noiseless device at 4096 shots would not return purity exactly 1, because finite-shot noise and the eigenvalue-clip projection to a valid density matrix each nudge the estimate. A single run cannot separate these two contributions; raising the shot count shrinks the sampling part, progressively isolating the genuine physical (decoherence) gap. Offline this reflects the modeled FakeManila noise; flip `USE_HARDWARE=True` and the very same plot shows the literal device's state.

### A real run on IBM hardware

On **`ibm_marrakesh`** (2026-05-30), single-qubit tomography of `|+>` reconstructed a state with **purity 0.974** and **fidelity 0.986** to the ideal — both below 1 (decoherence together with finite-shot and projection bias), yet markedly closer to pure than the offline FakeManila model (purity about 0.93). A modern device is a cleaner preparer. Set `USE_HARDWARE = True` to reproduce.

## 2 · The dictionary grows

Throughout the course we have kept a running dictionary translating each ideal object into what a real device hands back. Tomography lets us add the row that closes module 01's density-matrix arc:

| Ideal object | What a real device returns |
|---|---|
| pure state $|\psi\rangle$ | mixed state `rho_hat` (purity < 1) |
| exact `rho` | tomographic reconstruction from finite shots |
| fidelity 1 with the target | fidelity < 1 — the decoherence gap |

Module 01 introduced the density matrix as the language for imperfect, mixed states — exactly the situation a real qubit lives in. Here you watched that language earn its place: the device did not return a pure `|+>`, it returned a mixed `rho_hat`, and fidelity gave you a single number for how far the two stand apart.

## Your turn

- **Tomograph `|0>` instead.** Set `prep = QuantumCircuit(1)` with no gate (or `prep.id(0)`) and re-run. Expect a purity nearer 1: preparing `|0>` needs no state-prep gate, so it escapes the Hadamard's gate error entirely, and the device's read-out is most accurate on the `0` outcome that `|0>` overwhelmingly produces — both factors push the reconstructed state closer to pure.
- **Flip `USE_HARDWARE = True`** (with credentials set per `05/01`) and re-run. The purity and fidelity gaps you read are now the real chip's, not a model's.
- **Watch the gap grow.** Compute `1 - fidelity` and track it as you add gates before tomography — append an `h` then an `id`, or a short string of gates (note: two H gates compose to identity, so the *target stays the same* while the accumulated gate error grows) — and see the infidelity climb as the circuit lengthens. That growing `1 - fidelity` is a preview of the Bures sweep you build in `05/05`.

## What you built

- A full single-qubit tomography run on the primitive path: three Pauli-basis circuits transpiled, sampled with `SamplerV2`, and reconstructed into a density matrix `rho_hat` with `density_from_counts`.
- Two scores for the device's state: a purity below 1 that exposes it as mixed, and a fidelity below 1 that measures the gap to the ideal `|+>`.
- A new dictionary row tying the ideal `rho` you asked for to the device `rho_hat` you got — the decoherence gap made quantitative.

## What's next

Next you lift these ideas to two qubits and measure **quantum mutual information** on real hardware — the information one qubit's tomography reveals about another, and the first information-geometric quantity of the module to meet a live device.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, ch. 8.4 (state tomography) and ch. 9 (fidelity and distance measures), Cambridge University Press (2010).
- D. F. V. James, P. G. Kwiat, W. J. Munro & A. G. White, "Measurement of qubits," *Phys. Rev. A* **64**, 052312 (2001), doi:10.1103/PhysRevA.64.052312.

**Previous:** `notebooks/05_RealQuantumHardware/02_born_rule_on_real_hardware.ipynb` · **Next:** `notebooks/05_RealQuantumHardware/04_quantum_mutual_information_on_real_hardware.ipynb`